In [ ]:
def transform_entities(data):
    all_entities = []

    # Extract entities and probabilities from different categories
    for category, entities in data['entities'].items():
        if data['prob'][category]:
            for entity, prob in zip(entities, data['prob'][category]):
                all_entities.append({
                    'entity_group': category.capitalize(),
                    'score': prob[1],
                    'word': data['abstract'][entity['start']:entity['end']+1],  # Adjust slicing here
                    'start': entity['start'],
                    'end': entity['end']
                })

    # Sort entities by start, end, and -score (negative score for descending order)
    all_entities.sort(key=lambda x: (x['start'], x['end'], -x['score']))

    # Filter out overlapping entities, keeping only the one with the highest score
    non_overlapping_entities = []
    last_end = -1

    for entity in all_entities:
        if entity['start'] >= last_end:
            non_overlapping_entities.append(entity)
            last_end = entity['end']

    return non_overlapping_entities

# Example data input
data = {
    'pmid': '6b3838a5a74c4caa39f6071125ac7149f3c15cee940ed1034bec68b6',
    'entities': {
        'disease': [{'start': 45, 'end': 51}],
        'drug': [],
        'gene': [{'start': 0, 'end': 3}],
        'species': [],
        'cell_line': [],
        'DNA': [{'start': 0, 'end': 3}, {'start': 10, 'end': 23}],
        'RNA': [],
        'cell_type': []
    },
    'title': '',
    'abstract': 'kras is a proto-oncogene involved in various cancers.',
    'prob': {
        'disease': [[{'start': 45, 'end': 51}, 0.9998258948326111]],
        'drug': [],
        'gene': [[{'start': 0, 'end': 3}, 0.9947220087051392]],
        'species': [],
        'cell_line': [],
        'DNA': [[{'start': 0, 'end': 3}, 0.8248917460441589], [{'start': 10, 'end': 23}, 0.8577249050140381]],
        'RNA': [],
        'cell_type': []
    },
    'num_entities': 4,
    'elapse_time': {'biomedner_elapse_time': 0.3346831798553467, 'ner_elapse_time': 0.33543848991394043},
    'error_code': 0,
    'error_message': ''
}

# Transforming the example data
transformed_data = transform_entities(data)
print(transformed_data)


In [6]:
from Bio import Entrez

# Always provide your email address
Entrez.email = "your_email@example.com"

# Define the database and search term
db = "gene"
search_term = "Homo sapiens[orgn]"

# First, perform the search to get total count of records
search_handle = Entrez.esearch(db=db, term=search_term, usehistory="y", retmax=1)
search_results = Entrez.read(search_handle)
search_handle.close()

# Total count of records found
count = int(search_results['Count'])
print(f"Total records: {count}")

# Use history and WebEnv to fetch batches of records
webenv = search_results['WebEnv']
query_key = search_results['QueryKey']

batch_size = 1000 # NCBI typically allows batches of 500
gene_ids = []

# Retrieve all records in batches
for start in range(0, count, batch_size):
    fetch_handle = Entrez.efetch(db=db, rettype="uilist", retmode="text", retstart=start,
                                 retmax=batch_size, webenv=webenv, query_key=query_key)
    batch_ids = fetch_handle.read().split()
    fetch_handle.close()
    gene_ids.extend(batch_ids)
    print(f"Fetched batch from {start} to {start + batch_size - 1}")

# Output the total number of gene IDs collected
print(f"Collected a total of {len(gene_ids)} Gene IDs.")


Total records: 359884
Fetched batch from 0 to 999
Fetched batch from 1000 to 1999
Fetched batch from 2000 to 2999
Fetched batch from 3000 to 3999
Fetched batch from 4000 to 4999
Fetched batch from 5000 to 5999
Fetched batch from 6000 to 6999
Fetched batch from 7000 to 7999
Fetched batch from 8000 to 8999
Fetched batch from 9000 to 9999
Fetched batch from 10000 to 10999
Fetched batch from 11000 to 11999
Fetched batch from 12000 to 12999
Fetched batch from 13000 to 13999
Fetched batch from 14000 to 14999
Fetched batch from 15000 to 15999
Fetched batch from 16000 to 16999
Fetched batch from 17000 to 17999
Fetched batch from 18000 to 18999
Fetched batch from 19000 to 19999
Fetched batch from 20000 to 20999
Fetched batch from 21000 to 21999
Fetched batch from 22000 to 22999
Fetched batch from 23000 to 23999
Fetched batch from 24000 to 24999
Fetched batch from 25000 to 25999
Fetched batch from 26000 to 26999
Fetched batch from 27000 to 27999
Fetched batch from 28000 to 28999
Fetched batch fr

In [34]:
import numpy as np
np.where(np.array(gene_ids)=='6482')

(array([6540]),)

In [1]:
from convert import pubtator2dict_list

In [4]:
pubtator2dict_list("input/0a6edf3dc68091397084504be9bf9ff50acebcd142c1a949e4fc4654.PubTator.PubTator")[0]["abstract"]

'cancer is a disease.'

In [30]:
import re
import medspacy
from spacy.tokens import Span
from spacy.util import filter_spans
from spacy.language import Language

def pregnancy_recognizer(text):
    med_nlp = medspacy.load()
    med_nlp.disable_pipe('medspacy_target_matcher')
    regex_pattern = r"(?ix)\b(?:pregn\w*|matern\w*|gestat\w*|lactat\w*|breastfeed\w*|prenat\w*|antenat\w*|postpartum|childbear\w*|parturient|conceiv\w*|obstetr\w*|fertil\w*|gravid\w*|perinat\w*|neonat\w*|postnatal|childbirth|delivery|birthing|expectant\ mother|nursing\ mother|puerperal|midwifery|reproductive\ health|expecting(\ a\ child|\ baby)?)\b"
    
    @Language.component("pregnancy-ner")
    def regex_pattern_matcher_for_pregnancy(doc):
        compiled_pattern = re.compile(regex_pattern)
        original_ents = list(doc.ents)
        mwt_ents = []
        
        for match in re.finditer(compiled_pattern, doc.text):
            start, end = match.span()
            span = doc.char_span(start, end)
            if span is not None:
                mwt_ents.append((span.start, span.end, span.text))
        
        for ent in mwt_ents:
            start, end, name = ent
            per_ent = Span(doc, start, end, label="pregnancy")  # Assigning the label "PREGNANCY"
            original_ents.append(per_ent)
        
        doc.ents = filter_spans(original_ents)
        return doc

    # Add the component to the pipeline
    med_nlp.add_pipe("pregnancy-ner")
    
    # Process the input text with the modified pipeline
    doc = med_nlp(text)
    ent_list =[] 
    for entity in doc.ents:
        ent_list.append({"entity_group" : entity.label_, 
                        "text" : entity.text, 
                        "start": entity.start_char, 
                        "end": entity.end_char})
    return ent_list


In [31]:
docs = pregnancy_recognizer("The patient is pregnant and expecting a child in 3 months.")

In [32]:
docs

[{'entity_group': 'pregnancy', 'text': 'pregnant', 'start': 15, 'end': 23},
 {'entity_group': 'pregnancy',
  'text': 'expecting a child',
  'start': 28,
  'end': 45}]

In [35]:
from transformers import AutoTokenizer, pipeline
import re

def process_bio_med_negation(model_path_or_name="bvanaken/clinical-assertion-negation-bert", text, entities):
    # Initialize tokenizer and classification pipeline
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu"), help="Device to use")
    tokenizer = AutoTokenizer.from_pretrained(model_path_or_name, model_max_length=512, truncation=True)
    ner_pipeline = pipeline("text-classification", model=model_path_or_name, tokenizer=tokenizer, device=device)

    # Helper function to check if an entity is negated
    def is_entity_negated(sentence, entity, classify):
        entity_text = entity["text"]
        sentence_with_entity = re.sub(rf'\b{re.escape(entity_text)}\b', f"[entity]{entity_text}[entity]", sentence)
        classification = classify(sentence_with_entity)[0]
        is_negated = classification['label'] == 'ABSENT'
        entity["is_negated"] = "yes" if is_negated else "no"
    
    # Process each entity to assert negation
    for entity in entities:
        is_entity_negated(text, entity, ner_pipeline)
    
    return entities

# Example usage:
entities = [{"text": "headache"}, {"text": "fever"}]
model_path_or_name = "bvanaken/clinical-assertion-negation-bert"
device = 0 # Assuming CUDA device 0
result_entities = process_bio_med_negation(model_path_or_name, device, "Patient reports fever but does not report headache.", entities)
print(result_entities)


[{'text': 'headache', 'is_negated': 'yes'}, {'text': 'fever', 'is_negated': 'no'}]


In [1]:
trans_table = str.maketrans({
'"': '',
'{': '(',
'}': ')'})

In [3]:
"the red fox jumps over the {lazy} dog".translate(trans_table)

'the red fox jumps over the (lazy) dog'

In [2]:
import pandas as pd

In [14]:
df = pd.read_csv("../../data/preprocessed_data/clinical_trials/NCT00027586_preprocessed.csv")
# Assuming 'df' is your DataFrame



In [35]:
def resolve_overlaps(entities, priority_groups):
    # Sort entities by start position, and by end position in descending order for ties
    entities_sorted = sorted(entities, key=lambda x: (x['start'], -x['end']))

    # This will hold the final list of non-overlapping entities
    accepted_entities = []

    # Iterate through sorted entities
    for current in entities_sorted:
        overlap = False
        # Check for overlap with previously accepted entities
        for i, accepted in enumerate(list(accepted_entities)):  # Using list copy to modify original while iterating
            if (accepted['start'] < current['end'] and current['start'] < accepted['end']):
                overlap = True
                if accepted['entity_group'] not in priority_groups:
                    # Replace non-priority entity with a priority one, if current is in priority
                    if current['entity_group'] in priority_groups:
                        accepted_entities[i] = current
                elif current['entity_group'] in priority_groups:
                    # Both are priority groups; keep the later one (current)
                    accepted_entities[i] = current
                break
        
        # If no overlap was found, simply add the current entity
        if not overlap:
            accepted_entities.append(current)

    return accepted_entities

# Example usage
entities = [
    {"text": "entity1", "entity_group": "A", "start": 0, "end": 6, "score": 0.95, "is_negated": False, "synonyms": ["syn1"]},
    {"text": "entity2", "entity_group": "B", "start": 5, "end": 15, "score": 0.90, "is_negated": False, "synonyms": ["syn2"]},
    {"text": "entity3", "entity_group": "C", "start": 8, "end": 12, "score": 0.85, "is_negated": False, "synonyms": ["syn3"]},
    {"text": "entity4", "entity_group": "D", "start": 7, "end": 16, "score": 0.88, "is_negated": False, "synonyms": ["syn4"]}
]

# Define priority for entity groups, entities from these groups are kept in case of overlap
priority_groups = ['A', 'C']

# Resolve overlaps
resolved_entities = resolve_overlaps(entities, priority_groups)
print(resolved_entities)


[{'text': 'entity1', 'entity_group': 'A', 'start': 0, 'end': 6, 'score': 0.95, 'is_negated': False, 'synonyms': ['syn1']}, {'text': 'entity3', 'entity_group': 'C', 'start': 8, 'end': 12, 'score': 0.85, 'is_negated': False, 'synonyms': ['syn3']}]


In [32]:
df = pd.read_csv("../../data/regex_variants.tsv", sep="\t", header=None)
df[0].values.tolist()

['protein_mutation',
 'chromosome_arm',
 'aberration',
 'demethylation',
 'expression',
 'frameshift',
 'inframe deletion',
 'fusion',
 'exon',
 'gain of function',
 'cna',
 'amplification',
 'deletion',
 'germline amplification',
 'germline deletion',
 'germline loh',
 'germline mutation',
 'inactivation',
 'indel',
 'insertion',
 'inhibitor',
 'loss',
 'knockdown',
 'loss of function',
 'loss of heterozygosity',
 'methylation',
 'mutation',
 'phosphorylation',
 'promoter demethylation',
 'promoter methylation',
 'promoter mutation',
 'protein expression',
 'protein overexpression',
 'protein underexpression',
 'overexpression',
 'underexpression',
 'single nucleotide polymorphism',
 'splice variant',
 'structural variant',
 'synonymous mutation',
 'switch of function',
 'translocation',
 'truncation',
 'wildtype',
 'codon',
 'codon mutation',
 'exon mutation',
 'exon deletion',
 'exon insertion',
 'deficiency',
 'rearrangement',
 'mutation']

In [38]:
priority_groups = ['disease', 'gene', 'drug', 'cell_type', 'cell_line', 'species']
df = pd.read_csv("../../data/regex_variants.tsv", sep="\t", header=None)
variants_list = df[0].values.tolist()
variants_list.extend(priority_groups)
priority_groups = variants_list

In [39]:
priority_groups

['protein_mutation',
 'chromosome_arm',
 'aberration',
 'demethylation',
 'expression',
 'frameshift',
 'inframe deletion',
 'fusion',
 'exon',
 'gain of function',
 'cna',
 'amplification',
 'deletion',
 'germline amplification',
 'germline deletion',
 'germline loh',
 'germline mutation',
 'inactivation',
 'indel',
 'insertion',
 'inhibitor',
 'loss',
 'knockdown',
 'loss of function',
 'loss of heterozygosity',
 'methylation',
 'mutation',
 'phosphorylation',
 'promoter demethylation',
 'promoter methylation',
 'promoter mutation',
 'protein expression',
 'protein overexpression',
 'protein underexpression',
 'overexpression',
 'underexpression',
 'single nucleotide polymorphism',
 'splice variant',
 'structural variant',
 'synonymous mutation',
 'switch of function',
 'translocation',
 'truncation',
 'wildtype',
 'codon',
 'codon mutation',
 'exon mutation',
 'exon deletion',
 'exon insertion',
 'deficiency',
 'rearrangement',
 'mutation',
 'disease',
 'gene',
 'drug',
 'cell_type'

In [44]:
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForTokenClassification

tokenizer = AutoTokenizer.from_pretrained("d4data/biomedical-ner-all")
model = AutoModelForTokenClassification.from_pretrained("d4data/biomedical-ner-all")

pipe = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple") # pass device=0 if using gpu
pipe("""Platelet count >= 100 x 10^9/L""")


[{'entity_group': 'Diagnostic_procedure',
  'score': 0.9999463,
  'word': 'platelet count',
  'start': 0,
  'end': 14},
 {'entity_group': 'Lab_value',
  'score': 0.6932218,
  'word': '>',
  'start': 15,
  'end': 16},
 {'entity_group': 'Lab_value',
  'score': 0.9930078,
  'word': '100 x 10 ^ 9 / l',
  'start': 18,
  'end': 30}]

In [47]:
def merge_similar_consecutive_entities(entities):
    if not entities:
        return []

    # Define groups of interest directly as a set for faster membership testing
    groups_of_interest = {"Lab_value", "Diagnostic_procedure", "Therapeutic_procedure", "Detailed_description"}
    combined_entities = []
    current_entity = entities[0]

    # Ensure the current entity meets basic structure requirements
    necessary_keys = {'word', 'entity_group', 'start', 'end'}
    if not all(key in current_entity for key in necessary_keys):
        raise ValueError("Each entity must have 'text', 'entity_group', 'start', and 'end' keys.")

    for next_entity in entities[1:]:
        if not all(key in next_entity for key in necessary_keys):
            raise ValueError("Each entity must have 'text', 'entity_group', 'start', and 'end' keys.")

        # Check if both entities are in the groups of interest and are consecutive or nearly so (gap up to 3 characters)
        if (current_entity['entity_group'] == next_entity['entity_group'] and
            current_entity['entity_group'] in groups_of_interest and
            0 <= next_entity['start'] - current_entity['end'] - 1 <= 3):

            # Merge entities by updating the 'text' and 'end' of the current entity
            current_entity['word'] += ' ' + next_entity['word']
            current_entity['end'] = next_entity['end']
        else:
            # If not mergeable, add current to combined and move to next
            combined_entities.append(current_entity)
            current_entity = next_entity

    # Append the last processed entity
    combined_entities.append(current_entity)
    return combined_entities


In [48]:
_merge_similar_consecutive_entities(pipe("""Platelet count >= 100 x 10^9/L"""))

[{'entity_group': 'Diagnostic_procedure',
  'score': 0.9999463,
  'word': 'platelet count',
  'start': 0,
  'end': 14},
 {'entity_group': 'Lab_value',
  'score': 0.6932218,
  'word': '> 100 x 10 ^ 9 / l',
  'start': 15,
  'end': 30}]

In [3]:
import unicodedata
import re

def replace_unicode_symbols(text):
    def unicode_to_readable(match):
        char = match.group(0)
        try:
            name = unicodedata.name(char).lower() + ' '
            return name
        except ValueError:
            return char

    return re.sub(r'[\u0080-\uFFFF]', unicode_to_readable, text)

# Example usage
text = "Absolute Neutrophil Count \u22651.0 x 10^9/L"
transformed_text = replace_unicode_symbols(text)
print(transformed_text)



Absolute Neutrophil Count greater-than or equal to 1.0 x 10^9/L


In [4]:
text = "Females of childbearing potential must not be breast feeding and must have a negative serum or urine pregnancy test within 7 days of starting of treatment. The patient must agree to use adequate contraception for a minimum of two weeks prior to receiving study medication until 3 months after discontinuation of the study medication. Acceptable methods of contraception include total and true sexual abstinence, hormonal contraceptives that are not prone to drug-drug interactions {IUS Levonorgestrel Intra Uterine System {Mirena}, Medroxyprogesterone injections {Depo-Provera}}, copper-banded intra-uterine devices, and vasectomized partner. All hormonal methods of contraception should be used in combination with the use of a condom by their sexual male partner. Females of childbearing potential are defined as those who are not surgically sterile {i.e., bilateral tubal ligation, bilateral oophorectomy, or complete hysterectomy} or postmenopausal {defined as 12 months with no menses without an alternative medical cause}. Women will be considered post-menopausal if they have been amenorrheic for the past 12 months without an alternative medical cause. The following age-specific requirements must also apply: Women < 50 years old: they would be considered post-menopausal if they have been amenorrheic for the past 12 months or more following cessation of exogenous hormonal treatments. The levels of Luteinizing Hormone {LH} and Follicle-Stimulating Hormone {FSH} must also be in the post-menopausal range {as per the institution}. Women ≥ 50 years old: they would be considered post-menopausal if they have been amenorrheic for the past 12 months or more following cessation of all exogenous hormonal treatments, or have had radiation-induced oophorectomy with the last menses > 1 year ago, or have had chemotherapy-induced menopause with >1 year interval since last menses, or have had surgical sterilization by either bilateral oophorectomy or hysterectomy."

In [5]:
len(text.split())

286

In [12]:
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForTokenClassification

tokenizer = AutoTokenizer.from_pretrained("bvanaken/clinical-assertion-negation-bert", model_max_length=512,  truncation=True)
model = AutoModelForTokenClassification.from_pretrained("bvanaken/clinical-assertion-negation-bert")

pipe = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple") # pass device=0 if using gpu
pipe(text)


[{'entity_group': 'PRESENT',
  'score': 0.4324656,
  'word': 'females',
  'start': 0,
  'end': 7},
 {'entity_group': 'POSSIBLE',
  'score': 0.39169833,
  'word': 'of',
  'start': 8,
  'end': 10},
 {'entity_group': 'ABSENT',
  'score': 0.46005315,
  'word': 'child',
  'start': 11,
  'end': 16},
 {'entity_group': 'POSSIBLE',
  'score': 0.36947262,
  'word': '##bearing',
  'start': 16,
  'end': 23},
 {'entity_group': 'PRESENT',
  'score': 0.4090194,
  'word': 'potential',
  'start': 24,
  'end': 33},
 {'entity_group': 'POSSIBLE',
  'score': 0.41966844,
  'word': 'must not be',
  'start': 34,
  'end': 45},
 {'entity_group': 'ABSENT',
  'score': 0.37129334,
  'word': 'breast',
  'start': 46,
  'end': 52},
 {'entity_group': 'PRESENT',
  'score': 0.36750892,
  'word': 'feeding',
  'start': 53,
  'end': 60},
 {'entity_group': 'ABSENT',
  'score': 0.35546342,
  'word': 'and',
  'start': 61,
  'end': 64},
 {'entity_group': 'POSSIBLE',
  'score': 0.4393056,
  'word': 'must have a negative',
  'st

In [1]:
from transformers import AutoTokenizer, pipeline
import re

def process_bio_med_negation(text, entities):
    # Initialize tokenizer and classification pipeline
    device = "cuda:7"
    model = "bvanaken/clinical-assertion-negation-bert"
    tokenizer = AutoTokenizer.from_pretrained(model, model_max_length=512, truncation=True)
    ner_pipeline = pipeline("text-classification", model=model, tokenizer=tokenizer, device=device)

    # Run the classification pipeline on the sentence
    classification = ner_pipeline(text)[0]
    is_negated = classification['label'] == 'ABSENT'

    # Process each entity to assert negation
    for entity in entities:
        entity_text = entity["text"]
        if re.search(rf'\b{re.escape(entity_text)}\b', text):
            entity["is_negated"] = "yes" if is_negated else "no"
        else:
            entity["is_negated"] = "not found"
    
    return entities

# Example usage
text = "The patient has no signs of diabetes or hypertension."
entities = [{"text": "diabetes"}, {"text": "hypertension"}]
result = process_bio_med_negation(text, entities)
print(result)


/home/mabdallah/miniconda3/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/mabdallah/miniconda3/lib/python3.11/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


[{'text': 'diabetes', 'is_negated': 'yes'}, {'text': 'hypertension', 'is_negated': 'yes'}]


In [2]:
from concurrent.futures import ThreadPoolExecutor, as_completed